In [2]:
# Uncomment and run this cell if you're on Colab or Kaggle
!git clone https://github.com/nlp-with-transformers/notebooks.git
%cd notebooks

# Install the necessary requirements manually to avoid the installation script error
!pip install -q transformers datasets bertviz

Cloning into 'notebooks'...
remote: Enumerating objects: 542, done.
remote: Counting objects: 100% (254/254), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 542 (delta 219), reused 196 (delta 194), pack-reused 288 (from 1)
Receiving objects: 100% (542/542), 34.59 MiB | 11.01 MiB/s, done.
Resolving deltas: 100% (259/259), done.
Updating files: 100% (127/127), done.
/content/notebooks
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.5/157.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 4.5 MB/s eta 0:00:00


In [3]:
from transformers import AutoTokenizer
from bertviz.transformers_neuron_view import BertModel, BertTokenizer
from bertviz.neuron_view import show

model_ckpt = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = BertModel.from_pretrained(model_ckpt)
text = "time flies by like a plane"
show(model, "bert", tokenizer, text, display_mode="light", layer=0, head=8)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

100%|██████████| 440473133/440473133 [00:25<00:00, 17399300.17B/s]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
inputs = tokenizer(text, return_tensors='pt',add_speical_tokens=False)
inputs.input_ids

tensor([[  101,  2051, 10029,  2011,  2066,  1037,  4946,   102]])

In [5]:
from torch import nn
from transformers import AutoConfig

config = AutoConfig.from_pretrained(model_ckpt)
token_emb = nn.Embedding(config.vocab_size, config.hidden_size)
token_emb

Embedding(30522, 768)

In [6]:
inputs_embeds = token_emb(inputs.input_ids)
print(inputs_embeds.size())
inputs_embeds

torch.Size([1, 8, 768])


tensor([[[ 0.3868, -0.6945,  1.6036,  ..., -0.2335, -1.7456,  0.3305],
         [ 0.9813, -0.2404, -0.2502,  ...,  2.5403, -0.2900,  0.9436],
         [-1.3492,  0.6908, -0.9676,  ..., -0.7028, -2.2593, -0.3508],
         ...,
         [ 0.0498,  0.8913, -0.2509,  ..., -0.5947,  0.9100, -1.2287],
         [-1.7815, -0.7796,  1.0056,  ...,  1.6322,  1.1590, -0.6704],
         [ 0.6492,  0.0853, -1.6025,  ..., -0.9377, -0.3226, -0.9359]]],
       grad_fn=<EmbeddingBackward0>)

In [7]:
import torch
from math import sqrt

query = key = value = inputs_embeds
dim_k = key.size(-1)
scores =  torch.bmm(query, key.transpose(1,2)) / sqrt(dim_k)
print(scores.size())

torch.Size([1, 8, 8])


In [8]:
import torch.nn.functional as F

weights = F.softmax(scores, dim=-1)
weights.sum(dim=-1)

tensor([[1., 1., 1., 1., 1., 1., 1., 1.]], grad_fn=<SumBackward1>)

In [9]:
attn_output = torch.bmm(weights, value)
attn_output.shape

torch.Size([1, 8, 768])

In [10]:
def scaled_dot_product_attention(query,key,value):
  dim_k = query.size(-1)
  scores = torch.bmm(query, key.transpose(1,2)) / sqrt(dim_k)
  weights = F.softmax(scores,dim=-1)
  return torch.bmm(weights, value)

In [11]:
class AttentionHead(nn.Module):
  def __init__(self, embed_dim, head_dim):
    super().__init__()
    self.q = nn.Linear(embed_dim,head_dim)
    self.k = nn.Linear(embed_dim,head_dim)
    self.v = nn.Linear(embed_dim,head_dim)

  def forward(self, hidden_state):
    attn_output = scaled_dot_product_attention(self.q(hidden_state),self.k(hidden_state),self.v(hidden_state))
    return attn_output

In [12]:
class MultiHeadAttention(nn.Module):
  def __init__(self, config):
    super().__init__()
    embed_dim = config.hidden_size
    num_heads = config.num_attention_heads
    head_dim = embed_dim // num_heads
    self.heads = nn.ModuleList(
        [AttentionHead(embed_dim, head_dim) for _ in range(num_heads)]
    )
    self.output_linear = nn.Linear(embed_dim, embed_dim)
  def forward(self,hidden_state):
    x = torch.cat([h(hidden_state) for h in self.heads], dim = -1)
    x = self.output_linear(x)
    return x

In [13]:
multihead_attn = MultiHeadAttention(config)
attn_output = multihead_attn(inputs_embeds)
attn_output.size()

torch.Size([1, 8, 768])

In [14]:
#hide_output
from bertviz import head_view
from transformers import AutoModel

model = AutoModel.from_pretrained(model_ckpt, output_attentions=True)

sentence_a = "time flies like an arrow"
sentence_b = "fruit flies like a banana"

viz_inputs = tokenizer(sentence_a, sentence_b, return_tensors='pt')
attention = model(**viz_inputs).attentions
sentence_b_start = (viz_inputs.token_type_ids == 0).sum(dim=1)
tokens = tokenizer.convert_ids_to_tokens(viz_inputs.input_ids[0])

head_view(attention, tokens, sentence_b_start, heads=[8])

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


<IPython.core.display.Javascript object>

In [15]:
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.linear_1 = nn.Linear(config.hidden_size, config.intermediate_size)
        self.linear_2 = nn.Linear(config.intermediate_size, config.hidden_size)
        self.gelu = nn.GELU()
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(self, x):
        x = self.linear_1(x)
        x = self.gelu(x)
        x = self.linear_2(x)
        x = self.dropout(x)
        return x

In [16]:
feed_forward = FeedForward(config)
ff_outputs = feed_forward(attn_output)
ff_outputs.size()

torch.Size([1, 8, 768])

In [28]:
class TransformerEncoderLayer(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.layer_norm_1 = nn.LayerNorm(config.hidden_size)
    self.layer_norm_2 = nn.LayerNorm(config.hidden_size)
    self.attention = MultiHeadAttention(config)
    self.feed_forward = FeedForward(config)
  def forward(self, x):
    hidden_state = self.layer_norm_1(x)
    x = x + self.attention(hidden_state)
    x = x + self.feed_forward(self.layer_norm_2(x))
    return x

In [29]:
encoder_layer = TransformerEncoderLayer(config)
inputs_embeds.shape, encoder_layer(inputs_embeds).size()

(torch.Size([1, 8, 768]), torch.Size([1, 8, 768]))

In [34]:
class Embeddings(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.token_embeddings = nn.Embedding(config.vocab_size, config.hidden_size)
    self.position_embeddings = nn.Embedding(config.max_position_embeddings, config.hidden_size)
    self.layer_norm = nn.LayerNorm(config.hidden_size, config.hidden_size)
    self.dropout = nn.Dropout()

  def forward(self, input_ids):
    seq_length = input_ids.size(1)
    position_ids = torch.arange(seq_length , dtype=torch.long).unsqueeze(0)

    token_embeddings = self.token_embeddings(position_ids)
    position_embeddings = self.position_embeddings(position_ids)

    embeddings = token_embeddings + position_embeddings
    embeddings = self.layer_norm(embeddings)
    embeddigns = self.dropout(embeddings)
    return embeddings

In [35]:
embedding_layer = Embeddings(config)
embedding_layer(inputs.input_ids).size()

torch.Size([1, 8, 768])

In [47]:
class TransformersEncoder(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.embeddings = Embeddings(config)
    self.layers = nn.ModuleList([TransformerEncoderLayer(config) for _ in range(config.num_hidden_layers)])

  def forward(self, x):
    x = self.embeddings(x)
    for layer in self.layers:
      x = layer(x)
    return x

In [48]:
encoder = TransformersEncoder(config)
encoder(inputs.input_ids).size()

torch.Size([1, 8, 768])

In [51]:
class TransformerForSequenceClassification(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.encoder = TransformersEncoder(config)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

    def forward(self, x):
        x = self.encoder(x)[:, 0, :] # select hidden state of [CLS] token
        x = self.dropout(x)
        x = self.classifier(x)
        return x

In [52]:
config.num_labels = 3
encoder_classifier = TransformerForSequenceClassification(config)
encoder_classifier(inputs.input_ids).size()

torch.Size([1, 3])

In [53]:
seq_len = inputs.input_ids.size(-1)
mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0)
mask[0]

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])

In [54]:
scores.masked_fill(mask == 0, -float("inf"))

tensor([[[ 2.9212e+01,        -inf,        -inf,        -inf,        -inf,
                 -inf,        -inf,        -inf],
         [-9.3692e-01,  2.5098e+01,        -inf,        -inf,        -inf,
                 -inf,        -inf,        -inf],
         [-1.1333e+00,  3.7958e-01,  2.8531e+01,        -inf,        -inf,
                 -inf,        -inf,        -inf],
         [ 7.2520e-01,  5.9632e-02,  5.1684e-01,  2.9641e+01,        -inf,
                 -inf,        -inf,        -inf],
         [-3.2373e-01,  5.0474e-01, -3.4993e+00, -2.6530e+00,  2.5747e+01,
                 -inf,        -inf,        -inf],
         [ 7.7307e-02, -5.3067e-01, -3.5934e-01, -8.9934e-01,  6.5680e-01,
           2.6227e+01,        -inf,        -inf],
         [-1.6420e+00, -2.6428e+00, -8.6522e-02,  2.5028e+00, -1.1670e+00,
           1.0092e+00,  2.6584e+01,        -inf],
         [-1.2766e+00, -4.7276e-01,  4.1434e-01, -4.9646e-01,  3.2424e-01,
          -9.1690e-02, -2.5706e-02,  2.8257e+01]]]

In [55]:
def scaled_dot_product_attention(query, key, value, mask=None):
    dim_k = query.size(-1)
    scores = torch.bmm(query, key.transpose(1, 2)) / sqrt(dim_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    return weights.bmm(value)